# Lab05.2 – Traffic Sign Shape Detection

**Objective:** detect candidate signs and infer their geometric shape from contours.

Uses red/blue segmentation from Lab05.1.

**Step 1 — Preparation.** Load the libraries, configure `images/` and define `show_bgr()` for visualizing results. No image is processed yet.

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

IMAGE_DIR = Path('images')
def show_bgr(img, title='Image'):
    plt.figure(figsize=(11, 6))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.show()

print('OpenCV:', cv2.__version__)

**Step 2 — Colour-mask function.** Define HSV segmentation for red and blue signs. This function returns two binary images. The next cell displays them.

In [ ]:
def color_masks(bgr):
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    # OpenCV hue uses 0..179. Red wraps around zero.
    red1 = cv2.inRange(hsv, (0, 90, 70), (12, 255, 255))
    red2 = cv2.inRange(hsv, (168, 90, 70), (179, 255, 255))
    red_mask = cv2.bitwise_or(red1, red2)
    blue_mask = cv2.inRange(hsv, (95, 90, 60), (130, 255, 255))
    kernel = np.ones((3, 3), np.uint8)
    red_mask = cv2.morphologyEx(red_mask, cv2.MORPH_OPEN, kernel)
    blue_mask = cv2.morphologyEx(blue_mask, cv2.MORPH_OPEN, kernel)
    return {'red': red_mask, 'blue': blue_mask}

**Visual checkpoint — Colour masks.** The next cell reads the input and displays red and blue masks. If the segmentation is not correct, later contour and shape detection cannot succeed.

In [ ]:
img = cv2.imread(str(IMAGE_DIR / 'traffic_signs.png'))
assert img is not None, 'Missing images/traffic_signs.png'
masks = color_masks(img)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); axes[0].set_title('Input')
axes[1].imshow(masks['red'], cmap='gray', vmin=0, vmax=255); axes[1].set_title('Red mask')
axes[2].imshow(masks['blue'], cmap='gray', vmin=0, vmax=255); axes[2].set_title('Blue mask')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

**Step 4 — Contour extraction.** Define the contour finder and filtering rules: minimum area and bounding-box dimensions. A contour is a boundary of a connected region in a mask.

In [ ]:
def find_contours(mask):
    result = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return result[-2]  # works with OpenCV 3 and OpenCV 4

def candidates(bgr, min_area=400):
    out = []
    for color, mask in color_masks(bgr).items():
        for cnt in find_contours(mask):
            area = cv2.contourArea(cnt)
            if area < min_area: continue
            x, y, w, h = cv2.boundingRect(cnt)
            if w < 18 or h < 18: continue
            out.append({'color': color, 'contour': cnt,
                        'bbox': (x, y, w, h), 'area': area})
    return sorted(out, key=lambda c: (c['bbox'][1], c['bbox'][0]))

**Visual checkpoint — Candidate regions.** Highlight all accepted colour contours and their bounding boxes. These are only potential signs, not yet recognized shapes.

In [ ]:
preview = img.copy()
found = candidates(img)
for item in found:
    x, y, w, h = item['bbox']
    cv2.rectangle(preview, (x, y), (x + w, y + h), (0, 180, 0), 2)
show_bgr(preview, 'Colour-based sign candidates')
print('Candidates:', len(found))

**Step 6 — Geometric shape function.** Approximate a contour by a polygon and use the vertex count and circular fill ratio to assign a geometric shape. The result becomes visible in the final annotated image.

In [ ]:
def shape_of(contour):
    perimeter = cv2.arcLength(contour, True)
    if perimeter == 0: return 'unknown', 0
    approx = cv2.approxPolyDP(contour, 0.035 * perimeter, True)
    vertices = len(approx)
    if vertices == 3: return 'triangle', vertices
    if vertices == 4: return 'quadrilateral', vertices
    (_, _), radius = cv2.minEnclosingCircle(contour)
    fill_ratio = cv2.contourArea(contour) / (np.pi * radius * radius)
    if fill_ratio > 0.96: return 'circle', vertices
    if 7 <= vertices <= 10: return 'octagon', vertices
    return 'other', vertices

**Step 8 — Final detections.** For each candidate, determine its shape, draw a bounding box and label, then show the annotated image and print measurements.

In [ ]:
img = cv2.imread(str(IMAGE_DIR / 'traffic_signs.png'))
assert img is not None, 'Image missing'
found = candidates(img)
annotated = img.copy()
for c in found:
    shape, vertices = shape_of(c['contour'])
    x, y, w, h = c['bbox']
    cv2.rectangle(annotated, (x, y), (x+w, y+h), (40, 160, 40), 2)
    cv2.putText(annotated, shape, (x, max(15,y-8)), cv2.FONT_HERSHEY_SIMPLEX, .55, (0,0,0), 2)
    print(c['color'], shape, 'vertices:', vertices, 'bbox:', c['bbox'])
show_bgr(annotated, 'Detected shapes')

## Student tasks
1. Vary the `approxPolyDP` factor from 0.035 to 0.02 and 0.06. What changes?
2. Explain why the speed-limit sign is approximately circular even though its red region is a ring.
3. Vary `min_area` and observe which candidates remain.

**Limitation:** shape detection is sensitive to perspective, occlusion and low resolution.